# Sistema Massa–Molla–Amortidor: Resposta amb Laplace

Aquest notebook mostra com resoldre un sistema massa–molla–amortidor utilitzant la **transformada de Laplace**.

- **Massa ($m$)** [kg]
- **Amortidor ($c$)** [Ns/m]
- **Rigidesa ($k$)** [N/m]
- Força aplicada: $f(t) = 100·t$ N (lineal en el temps)

Es calcula la resposta temporal en **desplaçament i velocitat**, amb condicions inicials:

- $x(0) = x_0 = 0$ m  
- $v(0) = v_0 = 0.1$ m/s

També és possible **modificar $m$, $c$ i $k$ amb sliders** per veure l’efecte sobre la resposta.

In [1]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

sp.init_printing()

In [2]:
t, s = sp.symbols('t s')
f = sp.Function('f')(t)

In [3]:
def sistema_massa_molla_amortidor(m=3.0, c=200.0, k=1.0e4/3.0):
    # Input lineal
    f = 100*t

    # Transfer function
    G = 1/(m*s**2 + c*s + k)

    # Laplace input
    F = sp.laplace_transform(f, t, s, noconds=True)

    # Initial conditions
    x0 = 0.0
    v0 = 0.1
    Q = F + x0*m*s + m*v0 + c*x0

    # Laplace response
    X = sp.simplify(Q*G)
    Xd = s*X

    # Inverse Laplace
    U = sp.simplify(sp.inverse_laplace_transform(X, s, t))
    V = sp.simplify(sp.inverse_laplace_transform(Xd, s, t))

    # Time vector
    tt = np.arange(0, 0.5, 0.001)

    # Lambdify
    U_func = sp.lambdify(t, U, 'numpy')
    V_func = sp.lambdify(t, V, 'numpy')

    # Evaluate
    U_vals = U_func(tt)
    V_vals = V_func(tt)

    # Initial/final velocity
    v_initial = sp.limit(s*Xd, s, sp.oo)
    v_final = sp.limit(s*Xd, s, 0)

    # Subplots apilats: desplaçament i velocitat
    fig, axs = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

    # Desplaçament
    axs[0].plot(tt, U_vals, color='blue')
    axs[0].set_ylabel('Desplaçament (m)')
    axs[0].set_title(f'Resposta del sistema: m={m}kg, c={c}Ns/m, k={round(k,0)}N/m')
    axs[0].set_xlim(left=0)
    axs[0].grid(True)

    # Velocitat
    axs[1].plot(tt, V_vals, color='red')
    axs[1].axhline(float(v_final), color='orange', linestyle='--')
    axs[1].set_xlabel('Temps (s)')
    axs[1].set_ylabel('Velocitat (m/s)')
    axs[1].set_xlim(left=0)
    axs[1].grid(True)

    plt.tight_layout()
    plt.show()

    print(f"Velocitat inicial: {float(v_initial)} m/s, Velocitat final: {float(v_final)} m/s")

In [5]:
interact(sistema_massa_molla_amortidor,
         m=FloatSlider(value=3.0, min=1, max=10, step=0.5, description='Mass (kg)'),
         c=FloatSlider(value=200.0, min=50, max=800, step=50, description='Damping (Ns/m)'),
         k=FloatSlider(value=1.0e4/3.0, min=1200, max=1.0e4, step=500, description='Stiffness (N/m)'));


interactive(children=(FloatSlider(value=3.0, description='Mass (kg)', max=10.0, min=1.0, step=0.5), FloatSlide…